<a href="https://colab.research.google.com/github/GayathriSanthakumar/IMPROVING-ONLINE-LEARNING-OUTCOMES_ML/blob/main/MINOR_PROJECTipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import gradio as gr
import matplotlib.pyplot as plt


In [16]:
DATASET_FILE = "udemy_courses.csv"
COMPLETED_THRESHOLD = 70.0
WEIGHT_USER = 0.6
WEIGHT_COURSE = 0.4
N_RECOMMENDATIONS = 10
N_USERS = 200


In [17]:
df_courses = None
course_embeddings = None
df_user_history = None


In [18]:
from google.colab import files
uploaded = files.upload()


Saving udemy_courses 5.csv to udemy_courses 5 (1).csv


In [19]:
def init_data():
    global df_courses, course_embeddings, df_user_history

    dataset_file = list(uploaded.keys())[0]
    df_courses = pd.read_csv(dataset_file)

    # ✅ CHANGE 1: Added content_duration column validation
    required_cols = [
        "course_id",
        "course_title",
        "subject",
        "level",
        "num_subscribers",
        "content_duration"
    ]

    for col in required_cols:
        if col not in df_courses.columns:
            raise ValueError(f"Missing column: {col}")

    # -------------------------------------------------
    # COURSE TEXT FOR BERT EMBEDDINGS
    # -------------------------------------------------
    df_courses["text"] = (
        df_courses["course_title"].astype(str) + " " +
        df_courses["subject"].astype(str) + " " +
        df_courses["level"].astype(str)
    )

    model = SentenceTransformer("all-MiniLM-L6-v2")
    course_embeddings = model.encode(df_courses["text"].tolist())

    # -------------------------------------------------
    # COURSE DIFFICULTY PROXY
    # -------------------------------------------------
    max_sub = df_courses["num_subscribers"].max()
    df_courses["Intrinsic_Dropout_Rate"] = 100 - (
        (df_courses["num_subscribers"] / max_sub) * 100
    )

    # -------------------------------------------------
    # SYNTHETIC USER BEHAVIOR GENERATION
    # -------------------------------------------------
    np.random.seed(42)
    history = []

    for user_id in range(1, N_USERS + 1):

        n_courses = np.random.randint(6, 18)
        courses = np.random.choice(df_courses["course_id"], n_courses, replace=False)

        learner_type = np.random.choice(
            ["strong", "average", "weak"],
            p=[0.3, 0.4, 0.3]
        )

        for c in courses:
            if learner_type == "strong":
                completion = np.random.uniform(70, 100)
            elif learner_type == "average":
                completion = np.random.uniform(40, 90)
            else:
                completion = np.random.uniform(10, 60)

            history.append({
                "User_ID": user_id,
                "Course_ID": c,
                "Actual_Completion (%)": completion
            })

    df_user_history = pd.DataFrame(history)

    # -------------------------------------------------
    # DROPOUT %
    # -------------------------------------------------
    df_user_history["Dropout (%)"] = (
        100 - df_user_history["Actual_Completion (%)"]
    )

    # -------------------------------------------------
    # ✅ CHANGE 2: MERGE CATEGORY + CONTENT DURATION
    # -------------------------------------------------
    df_user_history = df_user_history.merge(
        df_courses[["course_id", "subject", "content_duration"]],
        left_on="Course_ID",
        right_on="course_id",
        how="left"
    )

    df_user_history.rename(columns={
        "subject": "Category",
        "content_duration": "Content_Duration"
    }, inplace=True)

    # -------------------------------------------------
    # ✅ CHANGE 3: CATEGORY SUCCESS RATE
    # -------------------------------------------------
    category_success = (
        df_user_history
        .groupby("Category")["Actual_Completion (%)"]
        .mean()
    )

    df_user_history["Category_Success_Rate (%)"] = (
        df_user_history["Category"].map(category_success)
    )

    # -------------------------------------------------
    # ✅ CHANGE 4: DROPOUT STAGE DETECTION
    # -------------------------------------------------
    def dropout_stage(row):
        progress = row["Actual_Completion (%)"]

        if progress >= 100:
            return "Completed"
        elif progress <= 20:
            return "Dropped in First 20%"
        elif progress >= 80:
            return "Dropped in Last 20%"
        else:
            return "Dropped in Middle"

    df_user_history["Dropout_Stage"] = (
        df_user_history.apply(dropout_stage, axis=1)
    )

    # -------------------------------------------------
    # ✅ CHANGE 5: INTELLIGENT DROPOUT REASON INFERENCE
    # -------------------------------------------------
    def infer_dropout_reason(row):

        completion = row["Actual_Completion (%)"]
        category_success = row["Category_Success_Rate (%)"]
        stage = row["Dropout_Stage"]

        if stage == "Completed":
            return "Successfully Completed"

        # Early Drop
        if stage == "Dropped in First 20%":

            if completion < category_success - 20:
                return "Possible mismatch between learner expectation and course content"

            elif category_success < 50:
                return "Category shows lower overall engagement among learners"

            else:
                return "Initial engagement difficulty or unclear course introduction"

        # Middle Drop
        elif stage == "Dropped in Middle":

            if completion < category_success - 15:
                return "Course difficulty progression may not match learner preparedness"

            elif category_success < 50:
                return "Category historically shows moderate completion challenges"

            else:
                return "Gradual disengagement due to increasing complexity"

        # Late Drop
        elif stage == "Dropped in Last 20%":

            if category_success > 70:
                return "Likely external constraints such as time availability"

            else:
                return "Final modules or assessments may present completion barriers"

        else:
            return "General disengagement pattern detected"

    df_user_history["Dropout_Reason"] = (
        df_user_history.apply(infer_dropout_reason, axis=1)
    )

    # -------------------------------------------------
    print("Dataset Loaded Successfully")
    print("Courses:", len(df_courses))
    print("Users:", N_USERS)

    return True

In [20]:
def classify_learner(user_avg_risk):
    if user_avg_risk < 30:
        return "Low Risk Learner"
    elif user_avg_risk < 60:
        return "Moderate Risk Learner"
    else:
        return "High Risk Learner"


In [21]:
def get_dynamic_weights(user_avg_risk):
    if user_avg_risk > 60:
        return 0.8, 0.2   # be conservative
    elif user_avg_risk < 30:
        return 0.4, 0.6   # allow challenge
    else:
        return 0.6, 0.4


In [22]:
def recommend_course_with_risk(user_id):

    user_history = df_user_history[df_user_history["User_ID"] == user_id]

    if user_history.empty:
        return "User not found",0,0,pd.DataFrame(),pd.DataFrame()

    user_avg_risk = user_history["Dropout (%)"].mean()
    learner_type = classify_learner(user_avg_risk)
    wu, wc = get_dynamic_weights(user_avg_risk)

    completed = user_history[
        user_history["Actual_Completion (%)"] >= COMPLETED_THRESHOLD
    ]

    # Cold-start handling
    if completed.empty:
        recs = df_courses.sort_values("Intrinsic_Dropout_Rate").head(10)

        message = f"""
### 🥇 Safe Starter Courses

**Learner Profile:** {learner_type}
**Risk Level:** High (Cold Start Detected)

Since no completed courses were found, low-risk courses are recommended.
"""

        history_display = user_history[
            ["Course_ID","Dropout (%)","Dropout_Stage","Dropout_Reason"]
        ]

        return message, user_avg_risk, user_avg_risk, history_display, recs

    vectors = []
    for cid in completed["Course_ID"]:
        idx = df_courses.index[df_courses["course_id"] == cid][0]
        vectors.append(course_embeddings[idx])

    user_vector = np.mean(vectors, axis=0).reshape(1,-1)

    sims = cosine_similarity(user_vector, course_embeddings)[0]
    sim_series = pd.Series(sims, index=df_courses["course_id"])

    taken = user_history["Course_ID"].tolist()
    sim_series = sim_series.drop(taken, errors="ignore")

    top_ids = sim_series.nlargest(N_RECOMMENDATIONS*5).index
    recs = df_courses[df_courses["course_id"].isin(top_ids)].copy()

    recs["Similarity_Score"] = sim_series.loc[recs["course_id"]].values

    recs["Predicted_Dropout_Risk (%)"] = (
        wu * user_avg_risk + wc * recs["Intrinsic_Dropout_Rate"]
    )

    recs["Recommendation_Score"] = (
        recs["Similarity_Score"] -
        recs["Predicted_Dropout_Risk (%)"]/100
    )

    recs = recs.sort_values("Recommendation_Score", ascending=False)
    best = recs.iloc[0]

    # Risk Level Label
    if best["Predicted_Dropout_Risk (%)"] < 30:
        risk_label = "Low Risk"
    elif best["Predicted_Dropout_Risk (%)"] < 60:
        risk_label = "Moderate Risk"
    else:
        risk_label = "High Risk"

    # Explain why this course is safer
    explanation = f"""
### 🧠 Why This Course Is Recommended

- Matches user's previously completed subjects
- Predicted dropout risk ({best['Predicted_Dropout_Risk (%)']:.2f}%) is adjusted using dynamic risk weighting
- Based on user's historical average dropout of {user_avg_risk:.2f}%
"""

    message = f"""
### 🥇 Best Personalized Course: {best['course_title']}

**Learner Profile:** {learner_type}
**Predicted Risk Level:** {risk_label}
**Dynamic Risk Weights:** User={wu}, Course={wc}

| Metric | Value |
|------|------|
| Subject | {best['subject']} |
| Level | {best['level']} |
| Similarity | {best['Similarity_Score']:.4f} |
| Predicted Risk | {best['Predicted_Dropout_Risk (%)']:.2f}% |

{explanation}
"""

    history_display = user_history[
        ["Course_ID","Dropout (%)","Dropout_Stage","Dropout_Reason"]
    ]

    top10 = recs.head(10)[
        ["course_title","subject","level",
         "Similarity_Score","Predicted_Dropout_Risk (%)"]
    ]

    return message, best["Predicted_Dropout_Risk (%)"], user_avg_risk, history_display, top10

In [23]:
def create_risk_meter(predicted,user_risk):
    fig,ax = plt.subplots(figsize=(4,1))
    ax.barh([0],[100],color="lightgray")
    ax.barh([0],[predicted],color="orange")
    ax.set_xlim(0,100)
    ax.set_yticks([])
    ax.set_title("Predicted Dropout Risk")
    ax.text(predicted,0,f"{predicted:.1f}%")
    return fig


In [24]:
def handle(uid):
    uid = int(uid)
    msg,pr,ur,hist,top10 = recommend_course_with_risk(uid)
    fig = create_risk_meter(pr,ur)
    return msg,ur,hist,top10,fig


In [25]:
if init_data():

    with gr.Blocks(theme=gr.themes.Soft(),
                   title="Risk-Adjusted Recommender") as demo:

        gr.Markdown("""
# Risk-Adjusted Course Recommendation Engine
### Adaptive & Risk-Aware MOOC Recommendation System
Enter User ID (1–200)
""")

        with gr.Row():
            uid = gr.Textbox(value="1",label="Enter User ID")
            btn = gr.Button("Analyze User & Recommend Course")

        gr.HTML("<hr>")

        with gr.Tabs():
            with gr.TabItem("🥇 Recommendation"):
                main_out = gr.Markdown()
                risk_plot = gr.Plot()
                user_prop = gr.Number(label="User Historical Dropout (%)")

            with gr.TabItem("📜 User History"):
                hist_out = gr.DataFrame()

            with gr.TabItem("📊 Top Courses"):
                top10_out = gr.DataFrame()

        btn.click(
            fn=handle,
            inputs=uid,
            outputs=[main_out,user_prop,hist_out,top10_out,risk_plot]
        )

    demo.launch(share=True)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Dataset Loaded Successfully
Courses: 3678
Users: 200


/tmp/ipython-input-3870512333.py:3: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(),


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://58ff763106b5715a4f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [26]:
print(df_user_history.columns)


Index(['User_ID', 'Course_ID', 'Actual_Completion (%)', 'Dropout (%)',
       'course_id', 'Category', 'Content_Duration',
       'Category_Success_Rate (%)', 'Dropout_Stage', 'Dropout_Reason'],
      dtype='object')


In [27]:
df_user_history.to_csv("user_history.csv", index=False)


In [28]:
from google.colab import files
files.download("user_history.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:
!git init
!git add .
!git commit -m "Initial commit - Risk Adjusted Course Recommender"
!git branch -M main
!git remote add origin https://github.com/GayathriSanthakumar/Risk-Adjusted-Course-Recommender.git
!git push -u origin main

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /content/.git/
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@bfff5aba8e26.(none)')
error: src refspec main does not match any
error: failed to push some refs to 'https://github.com/GayathriSanthakumar/Risk-Adjust

In [35]:
!git config --global user.name "Gayathri S"
!git config --global user.email "gayathrisanthakumar120506@gmail.com"

In [36]:
!git add .

In [37]:
!git commit -m "Added Risk-Aware Course Recommender System"

[main (root-commit) 2c57a1b] Added Risk-Aware Course Recommender System
 25 files changed, 60775 insertions(+)
 create mode 100644 .config/.last_opt_in_prompt.yaml
 create mode 100644 .config/.last_survey_prompt.yaml
 create mode 100644 .config/.last_update_check.json
 create mode 100644 .config/active_config
 create mode 100644 .config/config_sentinel
 create mode 100644 .config/configurations/config_default
 create mode 100644 .config/default_configs.db
 create mode 100644 .config/gce
 create mode 100644 .config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
 create mode 100644 .config/logs/2026.01.16/14.23.31.981136.log
 create mode 100644 .config/logs/2026.01.16/14.24.03.314209.log
 create mode 100644 .config/logs/2026.01.16/14.24.13.071214.log
 create mode 100644 .config/logs/2026.01.16/14.24.18.954466.log
 create mode 100644 .config/logs/2026.01.16/14.24.28.646070.log
 create mode 100644 .config/logs/2026.01.16/14.24.29.392089.log
 create mode 100644 .gradio/certi

In [39]:
!git branch -M main

In [41]:
!git remote set-url origin https://github.com/GayathriSanthakumar/IMPROVING-ONLINE-LEARNING-OUTCOMES_ML.git

In [43]:
!git push -u origin main

fatal: could not read Username for 'https://github.com': No such device or address
